# Triple Riding & Traffic Violation Detection — YOLO Training on Google Colab (T4 GPU)

**B.Tech Design Project-I — Alliance University**  
**Model Architecture:** Ultralytics YOLO (YOLO26n / YOLOv8n)  
**Target Classes (5 Classes):**
0. `MORE_THAN_TWO_PERSONS` (Triple Riding Violation)
1. `USING_MOBILE` (Driver Mobile Phone Violation)
2. `WITHOUT_HELMET` (Helmet Non-Compliance Violation)
3. `WITH_HELMET` (Compliant Helmet Rider)
4. `normal` (Standard Two-Wheeler Traffic)

---

### Step-by-Step Instructions:
1. **Select GPU Hardware Accelerator**:
   - Click **Runtime** in the top navigation bar → **Change runtime type**.
   - Under *Hardware accelerator*, select **T4 GPU** → Click **Save**.
2. **Run All Cells in Order** (`Ctrl + F9` or Shift + Enter per cell).
3. At the end of training, the notebook automatically saves and downloads `best.pt` to your computer.
4. Place `best.pt` into your project at `Prototype/models/triple_riding_best.pt`.

## 1. Verify GPU Accelerator

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Install Ultralytics & Dependencies

In [ ]:
!pip install -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Download the Triple Riding Dataset Directly into Colab (High Speed)

In [ ]:
from roboflow import Roboflow

# Direct Roboflow integration for the 6,057 image Triple Riding Dataset
rf = Roboflow(api_key="3OEqKlaUTLNgrFZGJiv6")
project = rf.workspace("kashish").project("3riders")
version = project.version(2)
dataset = version.download("yolov8")

print(f"\nDataset ready at: {dataset.location}")

## 4. Train YOLO Model on Triple Riding

In [ ]:
from ultralytics import YOLO
import os

# Initialize with base pretrained weights (YOLO26n or YOLOv8n)
# If using YOLO26, specify yolo26n.pt; defaults cleanly to yolov8n.pt if yolo26 weights are building
base_model = "yolo26n.pt"
model = YOLO(base_model)

data_yaml_path = os.path.join(dataset.location, "data.yaml")

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    patience=8,
    device=0,
    project="triple_riding_run",
    name="yolo_triple_detector",
    plots=True,
    verbose=True
)

print("Training complete!")

## 5. Evaluate Performance on Held-Out Test Set

In [ ]:
from IPython.display import Image, display

# Validate best model
best_weights = "runs/detect/triple_riding_run/yolo_triple_detector/weights/best.pt"
val_model = YOLO(best_weights)
metrics = val_model.val(data=data_yaml_path, split="test")

print(f"\n--- Test Set Metrics ---")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# Display confusion matrix
cm_path = "runs/detect/triple_riding_run/yolo_triple_detector/confusion_matrix.png"
if os.path.exists(cm_path):
    display(Image(filename=cm_path))

## 6. Download Trained Weights to Your Local Project

In [ ]:
from google.colab import files

# Downloads the trained weights directly to your browser download folder
files.download(best_weights)
print("Downloaded best.pt. Rename/copy it to 'Prototype/models/triple_riding_best.pt'.")
